# Winam Gulf water hyacinth spatial-panel test model

This notebook builds a **spatially explicit monthly panel dataset** from already-classified water hyacinth GeoTIFFs in Google Drive.

It is designed as a **test-period prototype**, not the final dissertation model. It:

1. mounts Google Drive;
2. finds classified WH GeoTIFFs for a chosen period;
3. creates fixed 500 m or 1 km grid cells over Winam Gulf;
4. aggregates each classified raster to `grid cell × month`;
5. optionally merges monthly environmental covariates such as rainfall, wind speed, lake level, turbidity or chlorophyll-a;
6. creates lagged WH-cover terms;
7. fits a simple two-stage spatial panel model:
   - WH presence/absence;
   - WH proportional cover where WH is present;
8. exports panel tables, predictions and grid outputs.

The intended response variable is **monthly WH proportional cover per spatial unit**, not pixel-level WH occurrence.

## 1. Install packages

Run this once at the start of a fresh Colab runtime.

In [ ]:
# In Colab, uncomment and run this if packages are missing.
# This can take a few minutes.

!pip -q install rasterio geopandas pyproj shapely fiona pyogrio statsmodels scikit-learn tqdm

## 2. Imports and Google Drive mount

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd

from shapely.geometry import box
from tqdm.auto import tqdm

import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.windows import from_bounds, Window

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
)

import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Google Drive was not mounted automatically. If running outside Colab, this is expected.")
    print(e)

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 120)

## 3. User configuration

Edit this cell first.

Important assumptions:

- By default this notebook now reads the **classified GeoTIFFs and run logs produced by `Classifier_Full_Stack_PostExport_TimeSeries_v4.ipynb`**.
- `WH_CLASS_VALUES` is set to `[2]`, matching the classifier notebook's `FLOATING_CLASS_CODE` for floating plants / water hyacinth.
- `CLASSIFIER_SENSOR_FILTER` and `CLASSIFIER_PRODUCT_FILTER` choose which classifier output becomes the panel response. The default uses the S2 Route B model output only, so S1, rule-based rasters, and probability rasters are not accidentally averaged into the same monthly response.
- If you want to run the panel model from a manually curated folder instead, set `USE_CLASSIFIER_RUN_LOG = False` and point `CLASSIFIED_TIF_DIR` at that folder.
- The default grid CRS is **EPSG:32736**, UTM Zone 36S, which is suitable for the Winam Gulf area south of the equator.
- The default AOI is the same broad Winam Gulf rectangle used in your previous GEE workflow.


In [ ]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

# These defaults match Classifier_Full_Stack_PostExport_TimeSeries_v4.ipynb:
#   DRIVE_ROOT / "outputs" / "full_stack_batch"
CLASSIFIER_BATCH_OUTPUT_DIR = Path("/content/drive/MyDrive/Winam_RF_Training_Data/outputs/full_stack_batch")
CLASSIFIER_TABLE_DIR = CLASSIFIER_BATCH_OUTPUT_DIR / "tables"
CLASSIFIED_TIF_DIR = CLASSIFIER_BATCH_OUTPUT_DIR / "classified_geotiffs"
CLASSIFIER_RUN_LOG_GLOB = "winam_full_stack_run_log_*.csv"

OUTPUT_DIR = Path("/content/drive/MyDrive/WH_spatial_panel_test")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Prefer the classifier notebook's run log because it records exactly which
# classified rasters were generated/reused for each sensor and date window.
# Set False to discover classified GeoTIFFs directly from CLASSIFIED_TIF_DIR.
USE_CLASSIFIER_RUN_LOG = True

# Classifier-output selection. Defaults intentionally keep a single response per
# month by using only S2 model classifications. Use None to include all sensors,
# or set e.g. ["S1"] if you want the SCC/S1 panel instead.
CLASSIFIER_SENSOR_FILTER = ["S2"]       # None, ["S2"], ["S1"], or ["S2", "S1"]
CLASSIFIER_PRODUCT_FILTER = "model"     # "model", "rules", or "all"
PREFER_PATCH_CLEANED_S1 = True           # run-log model paths already point to patch-cleaned files when enabled

# Optional monthly environmental covariates.
# Expected format: one row per month, with a column called 'month'.
# Example columns:
# month,rainfall_mm,wind_speed_ms,lake_level_m,turbidity,chl_a
ENV_MONTHLY_CSV = None
# ENV_MONTHLY_CSV = Path("/content/drive/MyDrive/WH_drivers/monthly_covariates.csv")

# Optional spatial covariates by grid_id.
# Expected format: one row per grid_id, with a column called 'grid_id'.
# Example columns:
# grid_id,dist_river_m,fetch_m,depth_m,shelter_index
SPATIAL_COVARIATES_CSV = None
# SPATIAL_COVARIATES_CSV = Path("/content/drive/MyDrive/WH_drivers/grid_spatial_covariates.csv")

# ---------------------------------------------------------------------
# Test period
# ---------------------------------------------------------------------

TEST_START = "2021-01-01"
TEST_END = "2021-12-31"

# ---------------------------------------------------------------------
# Spatial design
# ---------------------------------------------------------------------

# Use 500 for a more detailed panel, or 1000 for a simpler/coarser test.
CELL_SIZE_M = 500

# UTM Zone 36S. Suitable for Winam Gulf because it is just south of the equator.
PANEL_CRS = "EPSG:32736"

# Broad Winam Gulf AOI in WGS84: min_lon, min_lat, max_lon, max_lat
AOI_BBOX_WGS84 = (34.0, -0.55, 34.9, 0.0)

# ---------------------------------------------------------------------
# Classified raster settings
# ---------------------------------------------------------------------

# Classifier_Full_Stack_PostExport_TimeSeries_v4.ipynb writes floating plants /
# WH-like vegetation as class code 2 (FLOATING_CLASS_CODE = 2). This is the
# response class aggregated into WH cover by the panel model.
WH_CLASS_VALUES = [2]

# The classifier notebook writes NODATA_VALUE = 255. Keep it here even if a
# GeoTIFF's nodata metadata is missing.
EXTRA_NODATA_VALUES = [255]

# Valid classifier classes. S2 uses 0-3; S1/SCC uses 0-2. Including 3 is safe for
# S1 because it simply will not occur.
VALID_CLASS_VALUES = [0, 1, 2, 3]

# Minimum valid pixels in a cell-month for that observation to be retained.
MIN_VALID_PIXELS_PER_CELL_MONTH = 10

# How to combine duplicate classified GeoTIFFs in the same month.
# With the default classifier filters, duplicates usually mean multiple snapshots
# in the same month. Options: "mean", "max"
DUPLICATE_MONTH_METHOD = "mean"

# ---------------------------------------------------------------------
# Model settings
# ---------------------------------------------------------------------

TRAIN_FRACTION = 0.75
RANDOM_STATE = 42


## 4. Helper functions

In [ ]:
def parse_month_from_filename(path):
    """
    Extract a monthly timestamp from a GeoTIFF filename.

    Handles common patterns:
    - YYYY-MM-DD
    - YYYY_MM_DD
    - YYYY-MM
    - YYYY_MM
    - YYYYMMDD
    - YYYYMM

    If the name contains a date range, the first date is used.
    """
    name = Path(path).stem

    patterns = [
        r"(?P<year>20\d{2})[-_](?P<month>\d{2})[-_](?P<day>\d{2})",
        r"(?P<year>20\d{2})(?P<month>\d{2})(?P<day>\d{2})",
        r"(?P<year>20\d{2})[-_](?P<month>\d{2})",
        r"(?P<year>20\d{2})(?P<month>\d{2})",
    ]

    for pat in patterns:
        m = re.search(pat, name)
        if m:
            year = int(m.group("year"))
            month = int(m.group("month"))
            return pd.Timestamp(year=year, month=month, day=1)

    return pd.NaT


def _normalise_classifier_sensor(sensor):
    if pd.isna(sensor):
        return None
    sensor = str(sensor).upper()
    if sensor in {"S1", "S1_SCC", "SCC"}:
        return "S1"
    if sensor == "S2":
        return "S2"
    return sensor


def _classifier_sensor_from_path(path):
    stem = Path(path).stem.lower()
    if stem.startswith("winam_s2_predictors_"):
        return "S2"
    if stem.startswith("winam_s1_predictors_") or stem.startswith("winam_s1_scc_predictors_"):
        return "S1"
    return None


def _classifier_product_from_path(path):
    stem = Path(path).stem.lower()
    if stem.endswith("_proba") or "_proba" in stem:
        return "probability"
    if stem.endswith("_local_rules"):
        return "rules"
    if "_local_" in stem:
        return "model"
    return "unknown"


def _path_is_classifier_classification(path):
    product = _classifier_product_from_path(path)
    return product in {"model", "rules"}


def _passes_classifier_filters(sensor, product):
    sensor = _normalise_classifier_sensor(sensor)
    if CLASSIFIER_SENSOR_FILTER is not None:
        allowed = {_normalise_classifier_sensor(s) for s in CLASSIFIER_SENSOR_FILTER}
        if sensor not in allowed:
            return False

    if CLASSIFIER_PRODUCT_FILTER != "all" and product != CLASSIFIER_PRODUCT_FILTER:
        return False

    return True


def _latest_classifier_run_log(table_dir=CLASSIFIER_TABLE_DIR, pattern=CLASSIFIER_RUN_LOG_GLOB):
    table_dir = Path(table_dir)
    candidates = sorted(table_dir.glob(pattern), key=lambda p: p.stat().st_mtime)
    return candidates[-1] if candidates else None


def _records_from_classifier_run_log(run_log_path):
    """Read classifier-notebook outputs from its batch run log."""
    run_log_path = Path(run_log_path)
    log = pd.read_csv(run_log_path)
    required = {"sensor", "start_date", "end_date", "status"}
    missing = required.difference(log.columns)
    if missing:
        raise ValueError(f"Classifier run log is missing required columns {sorted(missing)}: {run_log_path}")

    path_columns = []
    if CLASSIFIER_PRODUCT_FILTER in {"model", "all"} and "model_classification_tif" in log.columns:
        path_columns.append(("model_classification_tif", "model"))
    if CLASSIFIER_PRODUCT_FILTER in {"rules", "all"} and "rule_classification_tif" in log.columns:
        path_columns.append(("rule_classification_tif", "rules"))
    if not path_columns:
        raise ValueError(
            "No usable classification path columns were found in the classifier run log. "
            "Expected model_classification_tif and/or rule_classification_tif."
        )

    records = []
    completed = log[log["status"].astype(str).str.lower().eq("completed")].copy()
    for _, row in completed.iterrows():
        sensor = _normalise_classifier_sensor(row["sensor"])
        for col, product in path_columns:
            raw_path = row.get(col, "")
            if pd.isna(raw_path) or str(raw_path).strip() == "":
                continue
            path = Path(str(raw_path))
            if not _passes_classifier_filters(sensor, product):
                continue
            if product == "model" and not PREFER_PATCH_CLEANED_S1 and path.stem.endswith("_patch_cleaned"):
                uncleaned = path.with_name(path.name.replace("_patch_cleaned", ""))
                if uncleaned.exists():
                    path = uncleaned
            if not path.exists():
                print(f"Skipping missing classifier output from run log: {path}")
                continue
            month = pd.Timestamp(row["start_date"]).to_period("M").to_timestamp()
            records.append({
                "path": path,
                "month": month,
                "sensor": sensor,
                "product": product,
                "start_date": row["start_date"],
                "end_date": row["end_date"],
                "source_run_log": str(run_log_path),
            })
    return records


def _records_from_classifier_folder(folder):
    """Fallback discovery for classifier GeoTIFF folders when no run log is used."""
    folder = Path(folder)
    files = sorted(list(folder.rglob("*.tif")) + list(folder.rglob("*.tiff")))
    records = []
    for fp in files:
        if not _path_is_classifier_classification(fp):
            continue
        product = _classifier_product_from_path(fp)
        sensor = _classifier_sensor_from_path(fp)
        if not _passes_classifier_filters(sensor, product):
            continue
        month = parse_month_from_filename(fp)
        if pd.isna(month):
            continue
        records.append({
            "path": fp,
            "month": month,
            "sensor": sensor,
            "product": product,
            "start_date": None,
            "end_date": None,
            "source_run_log": None,
        })
    return records


def find_classified_tifs(folder, start, end):
    """
    Find classifier-notebook classified GeoTIFFs and retain those whose month
    falls within the requested test period.

    When USE_CLASSIFIER_RUN_LOG is True, the latest run log in CLASSIFIER_TABLE_DIR
    is used so probability rasters, stale ad-hoc files, and unselected sensors or
    products are not accidentally included in the panel response. If no run log is
    available, discovery falls back to CLASSIFIED_TIF_DIR.
    """
    folder = Path(folder)
    start = pd.Timestamp(start).to_period("M").to_timestamp()
    end = pd.Timestamp(end).to_period("M").to_timestamp()

    records = []
    run_log_path = None
    if USE_CLASSIFIER_RUN_LOG:
        run_log_path = _latest_classifier_run_log()
        if run_log_path is not None:
            print(f"Using classifier run log: {run_log_path}")
            records = _records_from_classifier_run_log(run_log_path)
        else:
            print(f"No classifier run log found in {CLASSIFIER_TABLE_DIR}; falling back to folder discovery.")

    if run_log_path is None:
        records = _records_from_classifier_folder(folder)

    records = [r for r in records if start <= r["month"] <= end]
    out = pd.DataFrame(records)
    if len(out) == 0:
        raise FileNotFoundError(
            f"No classifier GeoTIFFs found in {folder} between {start.date()} and {end.date()} "
            f"for sensor filter {CLASSIFIER_SENSOR_FILTER} and product filter {CLASSIFIER_PRODUCT_FILTER}. "
            "Run Classifier_Full_Stack_PostExport_TimeSeries_v4.ipynb first, or adjust the classifier-output filters."
        )

    out = out.drop_duplicates(["path", "month", "sensor", "product"]).sort_values(
        ["month", "sensor", "product", "path"]
    ).reset_index(drop=True)
    return out


def create_grid_from_bbox(aoi_bbox_wgs84, cell_size_m, panel_crs):
    """
    Create square grid cells over the AOI bounding box.

    The grid is generated in a projected CRS so each cell has a meaningful metric area.
    """
    aoi_wgs84 = gpd.GeoDataFrame(
        {"name": ["aoi"]},
        geometry=[box(*aoi_bbox_wgs84)],
        crs="EPSG:4326",
    )

    aoi_proj = aoi_wgs84.to_crs(panel_crs)
    xmin, ymin, xmax, ymax = aoi_proj.total_bounds

    xmin = np.floor(xmin / cell_size_m) * cell_size_m
    ymin = np.floor(ymin / cell_size_m) * cell_size_m
    xmax = np.ceil(xmax / cell_size_m) * cell_size_m
    ymax = np.ceil(ymax / cell_size_m) * cell_size_m

    xs = np.arange(xmin, xmax, cell_size_m)
    ys = np.arange(ymin, ymax, cell_size_m)

    cells = []
    for x in xs:
        for y in ys:
            geom = box(x, y, x + cell_size_m, y + cell_size_m)
            if geom.intersects(aoi_proj.geometry.iloc[0]):
                cells.append(geom)

    grid = gpd.GeoDataFrame(
        {"grid_id": np.arange(len(cells), dtype=np.int32)},
        geometry=cells,
        crs=panel_crs,
    )

    # Retain simple centroid-based spatial controls.
    cent = grid.geometry.centroid
    grid["x_km"] = cent.x / 1000.0
    grid["y_km"] = cent.y / 1000.0
    grid["cell_area_m2"] = grid.geometry.area

    return grid


def clamp_window(win, width, height):
    """
    Clamp a rasterio window to raster bounds.
    """
    col_off = max(0, int(np.floor(win.col_off)))
    row_off = max(0, int(np.floor(win.row_off)))
    col_max = min(width, int(np.ceil(win.col_off + win.width)))
    row_max = min(height, int(np.ceil(win.row_off + win.height)))

    if col_max <= col_off or row_max <= row_off:
        return None

    return Window(col_off, row_off, col_max - col_off, row_max - row_off)


def aggregate_one_raster_to_grid(
    tif_path,
    month,
    grid_gdf,
    panel_crs,
    wh_class_values,
    extra_nodata_values=None,
    valid_class_values=None,
):
    """
    Aggregate one classified WH GeoTIFF to the fixed grid.

    The raster is read through a WarpedVRT in the panel CRS, so pixel area is
    approximately metric and consistent across rasters.
    """
    extra_nodata_values = [] if extra_nodata_values is None else list(extra_nodata_values)
    wh_class_values = np.array(wh_class_values)

    with rasterio.open(tif_path) as src:
        vrt_kwargs = {
            "crs": panel_crs,
            "resampling": Resampling.nearest,
        }

        if src.nodata is not None:
            vrt_kwargs["nodata"] = src.nodata

        with WarpedVRT(src, **vrt_kwargs) as vrt:
            grid_bounds = grid_gdf.total_bounds
            raw_win = from_bounds(*grid_bounds, transform=vrt.transform)
            win = clamp_window(raw_win, vrt.width, vrt.height)

            if win is None:
                return pd.DataFrame()

            arr = vrt.read(1, window=win)
            transform = vrt.window_transform(win)

            # Rasterize grid IDs onto the same raster grid.
            shapes = ((geom, int(gid)) for geom, gid in zip(grid_gdf.geometry, grid_gdf["grid_id"]))
            id_arr = rasterize(
                shapes=shapes,
                out_shape=arr.shape,
                transform=transform,
                fill=-1,
                dtype="int32",
            )

            valid = id_arr >= 0

            if src.nodata is not None:
                valid &= arr != src.nodata

            if len(extra_nodata_values) > 0:
                valid &= ~np.isin(arr, np.array(extra_nodata_values))

            if valid_class_values is not None:
                valid &= np.isin(arr, np.array(valid_class_values))

            wh = valid & np.isin(arr, wh_class_values)

            valid_ids = id_arr[valid].astype(np.int32)
            wh_ids = id_arr[wh].astype(np.int32)

            n_cells = int(grid_gdf["grid_id"].max()) + 1
            valid_counts = np.bincount(valid_ids, minlength=n_cells)
            wh_counts = np.bincount(wh_ids, minlength=n_cells)

            pixel_area_m2 = abs(transform.a * transform.e)

            df = pd.DataFrame({
                "grid_id": np.arange(n_cells, dtype=np.int32),
                "month": pd.Timestamp(month),
                "valid_pixels": valid_counts.astype(np.int64),
                "wh_pixels": wh_counts.astype(np.int64),
            })

            df = df[df["valid_pixels"] > 0].copy()
            df["valid_area_m2"] = df["valid_pixels"] * pixel_area_m2
            df["wh_area_m2"] = df["wh_pixels"] * pixel_area_m2
            df["wh_cover"] = df["wh_area_m2"] / df["valid_area_m2"]
            df["source_file"] = Path(tif_path).name

            return df


def load_monthly_covariates(csv_path):
    """
    Load monthly covariates from CSV.

    Required column:
    - month

    The month column can be YYYY-MM, YYYY-MM-DD, or another pandas-parseable date.
    """
    df = pd.read_csv(csv_path)
    if "month" not in df.columns:
        raise ValueError("ENV_MONTHLY_CSV must contain a 'month' column.")

    df["month"] = pd.to_datetime(df["month"]).dt.to_period("M").dt.to_timestamp()
    return df


def clean_feature_columns(df, candidate_cols):
    """
    Keep numeric feature columns that are not entirely missing and not constant.
    """
    keep = []
    for col in candidate_cols:
        if col not in df.columns:
            continue
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue
        non_missing = df[col].dropna()
        if len(non_missing) == 0:
            continue
        if non_missing.nunique() <= 1:
            continue
        keep.append(col)
    return keep


## 5. Find classifier-output GeoTIFFs for the test period

This step uses the classifier notebook's latest batch run log when available. The default filters select only S2 model-classification rasters, excluding S1, paper-rule rasters, and `_proba` probability rasters so the monthly response is not mixed across products.


In [ ]:
tif_index = find_classified_tifs(CLASSIFIED_TIF_DIR, TEST_START, TEST_END)

print(f"Found {len(tif_index)} classifier GeoTIFF(s).")
display(tif_index.head(20))

if {"sensor", "product"}.issubset(tif_index.columns):
    print("Classifier outputs selected for the panel:")
    display(tif_index.groupby(["sensor", "product"]).size().rename("n_geotiffs").reset_index())

dupes = tif_index.groupby("month").size()
dupes = dupes[dupes > 1]
if len(dupes) > 0:
    print("Months with multiple GeoTIFFs. These will be combined later using:", DUPLICATE_MONTH_METHOD)
    display(dupes)


## 6. Create the spatial grid

This creates the fixed spatial units used in the panel. For a quick test, `CELL_SIZE_M = 1000` is faster. For a more detailed prototype, use `CELL_SIZE_M = 500`.

In [ ]:
grid = create_grid_from_bbox(AOI_BBOX_WGS84, CELL_SIZE_M, PANEL_CRS)

print(f"Created {len(grid):,} grid cells at {CELL_SIZE_M} m resolution.")
display(grid.head())

grid_path_gpkg = OUTPUT_DIR / f"winam_grid_{CELL_SIZE_M}m.gpkg"
grid_path_geojson = OUTPUT_DIR / f"winam_grid_{CELL_SIZE_M}m.geojson"

grid.to_file(grid_path_gpkg, layer="grid", driver="GPKG")
grid.to_crs("EPSG:4326").to_file(grid_path_geojson, driver="GeoJSON")

print("Saved grid to:")
print(grid_path_gpkg)
print(grid_path_geojson)

ax = grid.to_crs("EPSG:4326").plot(figsize=(8, 6), facecolor="none", edgecolor="black", linewidth=0.2)
ax.set_title(f"Winam Gulf spatial panel grid: {CELL_SIZE_M} m")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

## 7. Aggregate classifier-output WH rasters to the grid

This is the main preprocessing step.

For every selected classifier GeoTIFF, the notebook calculates, for each grid cell:

- valid classified pixel count;
- WH / floating-plant pixel count using `WH_CLASS_VALUES`;
- valid classified area;
- WH area;
- WH proportional cover.

The result is a `grid_id × month` panel with source classifier metadata retained for auditability.


In [ ]:
panel_parts = []
metadata_cols = ["sensor", "product", "start_date", "end_date", "source_run_log"]

for row in tqdm(tif_index.itertuples(index=False), total=len(tif_index)):
    df = aggregate_one_raster_to_grid(
        tif_path=row.path,
        month=row.month,
        grid_gdf=grid,
        panel_crs=PANEL_CRS,
        wh_class_values=WH_CLASS_VALUES,
        extra_nodata_values=EXTRA_NODATA_VALUES,
        valid_class_values=VALID_CLASS_VALUES,
    )
    if len(df) > 0:
        row_dict = row._asdict()
        for col in metadata_cols:
            if col in row_dict:
                df[f"source_{col}"] = row_dict[col]
        panel_parts.append(df)

if len(panel_parts) == 0:
    raise RuntimeError("No panel rows were created. Check raster CRS, AOI, class codes and no-data settings.")

panel_raw = pd.concat(panel_parts, ignore_index=True)

print(f"Raw panel rows before duplicate-month handling: {len(panel_raw):,}")
display(panel_raw.head())


### Combine duplicate observations within the same month

If the classifier output has only one selected raster per month, this step will not change much.

If there are multiple selected snapshots in the same month, the default behaviour is to take the **mean** grid-cell WH cover/area across those rasters. You can change this to `max` in the configuration cell.


In [ ]:
if DUPLICATE_MONTH_METHOD not in {"mean", "max"}:
    raise ValueError("DUPLICATE_MONTH_METHOD must be either 'mean' or 'max'.")

source_agg = lambda x: "; ".join(sorted(set(map(str, x.dropna()))))
source_metadata_cols = [c for c in panel_raw.columns if c.startswith("source_")]

if DUPLICATE_MONTH_METHOD == "mean":
    agg_funcs = {
        "valid_pixels": "mean",
        "wh_pixels": "mean",
        "valid_area_m2": "mean",
        "wh_area_m2": "mean",
        "wh_cover": "mean",
        "source_file": source_agg,
    }
elif DUPLICATE_MONTH_METHOD == "max":
    agg_funcs = {
        "valid_pixels": "max",
        "wh_pixels": "max",
        "valid_area_m2": "max",
        "wh_area_m2": "max",
        "wh_cover": "max",
        "source_file": source_agg,
    }

for col in source_metadata_cols:
    agg_funcs[col] = source_agg

panel = (
    panel_raw
    .groupby(["grid_id", "month"], as_index=False)
    .agg(agg_funcs)
)

# Retain only minimally valid cell-months.
panel = panel[panel["valid_pixels"] >= MIN_VALID_PIXELS_PER_CELL_MONTH].copy()

# Merge grid attributes.
grid_attrs = grid.drop(columns="geometry").copy()
panel = panel.merge(grid_attrs, on="grid_id", how="left")

# Basic response variables.
panel["wh_present"] = panel["wh_area_m2"] > 0
panel["wh_area_ha"] = panel["wh_area_m2"] / 10_000
panel["valid_area_ha"] = panel["valid_area_m2"] / 10_000

print(f"Panel rows after duplicate-month handling and validity filtering: {len(panel):,}")
print(f"Months: {panel['month'].nunique()}")
print(f"Grid cells represented: {panel['grid_id'].nunique()}")

display(panel.head())


## 8. Merge optional environmental and spatial covariates

For the final dissertation model, this is where the key environmental-driver variables enter.

The classified GeoTIFFs alone can produce the WH response variable, but they cannot explain environmental controls unless you merge additional variables.

Recommended monthly covariates:

- rainfall total;
- antecedent/cumulative rainfall;
- wind speed;
- directional/onshore wind component;
- lake-level anomaly;
- turbidity or suspended sediment proxy;
- chlorophyll-a or trophic proxy.

Recommended static spatial covariates:

- distance to river mouth;
- depth or distance from shore;
- fetch/shelter index;
- embayment/bay identifier;
- wetland adjacency.

In [ ]:
# Monthly covariates: one row per month
if ENV_MONTHLY_CSV is not None:
    env = load_monthly_covariates(ENV_MONTHLY_CSV)
    print("Loaded monthly environmental covariates:")
    display(env.head())

    panel = panel.merge(env, on="month", how="left")
else:
    print("No ENV_MONTHLY_CSV supplied.")
    print("The test model will use seasonality, simple grid coordinates and lagged WH cover only.")
    print("For final environmental inference, provide a monthly covariate CSV and rerun this notebook.")

# Spatial covariates: one row per grid_id
if SPATIAL_COVARIATES_CSV is not None:
    spatial_cov = pd.read_csv(SPATIAL_COVARIATES_CSV)
    if "grid_id" not in spatial_cov.columns:
        raise ValueError("SPATIAL_COVARIATES_CSV must contain a 'grid_id' column.")

    print("Loaded spatial covariates:")
    display(spatial_cov.head())

    # Avoid duplicating columns already present.
    overlap = [c for c in spatial_cov.columns if c in panel.columns and c != "grid_id"]
    if overlap:
        print("Dropping overlapping spatial covariate columns:", overlap)
        spatial_cov = spatial_cov.drop(columns=overlap)

    panel = panel.merge(spatial_cov, on="grid_id", how="left")
else:
    print("No SPATIAL_COVARIATES_CSV supplied.")
    print("The grid centroid coordinates x_km/y_km will act only as crude spatial controls.")

## 9. Feature engineering

This creates:

- cyclic seasonal terms;
- linear time index;
- lagged WH cover and presence per grid cell;
- optional one-month lags for environmental covariates.

Lagging helps because WH extent in a given month may respond to antecedent rainfall, wind or nutrient conditions, rather than only same-month conditions.

In [ ]:
panel = panel.sort_values(["grid_id", "month"]).reset_index(drop=True)

# Seasonal and time terms.
panel["month_num"] = panel["month"].dt.month
panel["month_sin"] = np.sin(2 * np.pi * panel["month_num"] / 12)
panel["month_cos"] = np.cos(2 * np.pi * panel["month_num"] / 12)

month_min = panel["month"].min()
panel["time_index"] = (
    (panel["month"].dt.year - month_min.year) * 12
    + (panel["month"].dt.month - month_min.month)
)

# Lagged WH terms by spatial unit.
panel["wh_cover_lag1"] = panel.groupby("grid_id")["wh_cover"].shift(1)
panel["wh_present_lag1"] = panel.groupby("grid_id")["wh_present"].shift(1).astype("float")

# Identify monthly environmental covariates if supplied.
base_non_feature_cols = {
    "grid_id", "month", "source_file",
    "valid_pixels", "wh_pixels", "valid_area_m2", "wh_area_m2",
    "wh_cover", "wh_present", "wh_area_ha", "valid_area_ha",
    "cell_area_m2", "month_num",
}

numeric_cols = [c for c in panel.columns if pd.api.types.is_numeric_dtype(panel[c])]
env_candidate_cols = [
    c for c in numeric_cols
    if c not in base_non_feature_cols
    and c not in {"x_km", "y_km", "time_index", "month_sin", "month_cos", "wh_cover_lag1", "wh_present_lag1"}
]

# Create one-month lags for environmental covariates.
# If these covariates are month-level only, this will still work because each grid cell receives the same monthly value.
for col in env_candidate_cols:
    panel[f"{col}_lag1"] = panel.groupby("grid_id")[col].shift(1)

print("Potential environmental/spatial numeric covariates detected:")
print(env_candidate_cols)

display(panel.head())

## 10. Exploratory checks

Before modelling, check whether WH observations are mostly zero, whether the time series looks plausible, and whether some months have poor coverage.

In [ ]:
monthly_summary = (
    panel.groupby("month", as_index=False)
    .agg(
        wh_area_ha=("wh_area_ha", "sum"),
        valid_area_ha=("valid_area_ha", "sum"),
        mean_cover=("wh_cover", "mean"),
        occurrence_rate=("wh_present", "mean"),
        n_cells=("grid_id", "nunique"),
    )
)

display(monthly_summary)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly_summary["month"], monthly_summary["wh_area_ha"], marker="o")
ax.set_title("Mapped WH area by month")
ax.set_xlabel("Month")
ax.set_ylabel("WH area aggregated across grid cells (ha)")
ax.grid(True, alpha=0.3)
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly_summary["month"], monthly_summary["occurrence_rate"], marker="o")
ax.set_title("Grid-cell WH occurrence rate by month")
ax.set_xlabel("Month")
ax.set_ylabel("Share of valid grid cells with WH present")
ax.grid(True, alpha=0.3)
plt.show()

print("Overall share of cell-months with WH present:", round(panel["wh_present"].mean(), 4))

## 11. Prepare model dataset

This model is intentionally simple and robust for a test period.

It uses a **temporally blocked train/test split**, rather than a random split, because random splitting would leak information from adjacent months into the test set.

In [ ]:
# Candidate model features.
candidate_features = [
    "month_sin",
    "month_cos",
    "time_index",
    "x_km",
    "y_km",
    "wh_cover_lag1",
    "wh_present_lag1",
]

# Add covariates and their one-month lags.
candidate_features += env_candidate_cols
candidate_features += [f"{c}_lag1" for c in env_candidate_cols if f"{c}_lag1" in panel.columns]

feature_cols = clean_feature_columns(panel, candidate_features)

print("Model feature columns:")
for c in feature_cols:
    print(" -", c)

model_df = panel.dropna(subset=["wh_cover", "wh_present"]).copy()

# Drop rows where lagged WH values are unavailable.
# For a very short test period this may remove the first month.
required_cols = feature_cols + ["wh_cover", "wh_present", "grid_id", "month", "valid_area_m2"]
model_df = model_df.dropna(subset=required_cols).copy()

if len(model_df) == 0:
    raise RuntimeError("No model rows remain after dropping missing feature/response values.")

months_sorted = np.array(sorted(model_df["month"].unique()))
split_idx = max(1, int(len(months_sorted) * TRAIN_FRACTION))
split_month = months_sorted[split_idx]

train = model_df[model_df["month"] < split_month].copy()
test = model_df[model_df["month"] >= split_month].copy()

print(f"Training months: {train['month'].min().date()} to {train['month'].max().date()}")
print(f"Testing months:  {test['month'].min().date()} to {test['month'].max().date()}")
print(f"Train rows: {len(train):,}")
print(f"Test rows:  {len(test):,}")
print(f"Train WH-present rate: {train['wh_present'].mean():.4f}")
print(f"Test WH-present rate:  {test['wh_present'].mean():.4f}")

X_train = train[feature_cols]
X_test = test[feature_cols]

y_train_presence = train["wh_present"].astype(int)
y_test_presence = test["wh_present"].astype(int)

## 12. Stage 1 model: WH presence/absence

This estimates whether each grid cell contains any mapped WH in each month.

For the final dissertation, you may want to compare this with a GAM, GLMM or Bayesian model. This notebook uses regularised logistic regression because it is fast, transparent and easy to test.

In [ ]:
if y_train_presence.nunique() < 2:
    raise RuntimeError(
        "The training set contains only one presence/absence class. "
        "Try a longer test period, larger AOI, different class codes, or coarser grid."
    )

presence_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

presence_model.fit(X_train, y_train_presence)

test_presence_prob = presence_model.predict_proba(X_test)[:, 1]
test_presence_pred = test_presence_prob >= 0.5

print("Presence/absence classification report:")
print(classification_report(y_test_presence, test_presence_pred, digits=3))

if y_test_presence.nunique() == 2:
    print("ROC AUC:", round(roc_auc_score(y_test_presence, test_presence_prob), 4))
    print("Average precision:", round(average_precision_score(y_test_presence, test_presence_prob), 4))
    print("Brier score:", round(brier_score_loss(y_test_presence, test_presence_prob), 4))
else:
    print("Test set contains only one class, so ROC AUC and average precision are undefined.")

presence_coef = pd.DataFrame({
    "feature": feature_cols,
    "standardised_logit_coefficient": presence_model.named_steps["model"].coef_[0],
}).sort_values("standardised_logit_coefficient", key=np.abs, ascending=False)

display(presence_coef)

## 13. Stage 2 model: WH cover where present

This estimates proportional WH cover for cell-months where WH is present.

The response is logit-transformed WH cover. Predictions are converted back to proportional cover.

In [ ]:
def logit_clip(x, eps=1e-5):
    x = np.clip(np.asarray(x, dtype=float), eps, 1 - eps)
    return np.log(x / (1 - x))


def inv_logit(z):
    return 1 / (1 + np.exp(-z))


positive_train = train[train["wh_present"]].copy()
positive_test = test[test["wh_present"]].copy()

if len(positive_train) < 10:
    raise RuntimeError(
        "Too few positive WH training rows for the cover model. "
        "Try a longer period, larger AOI, coarser grid, or check WH_CLASS_VALUES."
    )

X_train_cover = positive_train[feature_cols]
y_train_cover = logit_clip(positive_train["wh_cover"])

cover_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0, random_state=RANDOM_STATE)),
])

cover_model.fit(X_train_cover, y_train_cover)

# Conditional cover predictions for all test rows.
test_cover_logit_pred = cover_model.predict(X_test)
test_cover_conditional_pred = inv_logit(test_cover_logit_pred)

# Evaluate only where WH was truly present.
if len(positive_test) > 0:
    X_pos_test = positive_test[feature_cols]
    pos_pred = inv_logit(cover_model.predict(X_pos_test))

    rmse = mean_squared_error(positive_test["wh_cover"], pos_pred, squared=False)
    mae = mean_absolute_error(positive_test["wh_cover"], pos_pred)

    print("Positive-cover model performance, evaluated only where WH is observed present:")
    print("RMSE:", round(rmse, 4))
    print("MAE: ", round(mae, 4))
else:
    print("No positive WH rows in test set, so positive-cover performance is undefined.")

cover_coef = pd.DataFrame({
    "feature": feature_cols,
    "standardised_cover_coefficient": cover_model.named_steps["model"].coef_,
}).sort_values("standardised_cover_coefficient", key=np.abs, ascending=False)

display(cover_coef)

## 14. Combined expected-cover predictions

The two stages are combined as:

`expected WH cover = probability of WH presence × conditional WH cover`

This is useful for estimating expected WH area in each grid cell and month.

In [ ]:
test_preds = test[[
    "grid_id", "month", "valid_area_m2", "wh_area_m2", "wh_area_ha",
    "wh_cover", "wh_present"
]].copy()

test_preds["pred_presence_prob"] = test_presence_prob
test_preds["pred_cover_if_present"] = test_cover_conditional_pred
test_preds["pred_expected_cover"] = (
    test_preds["pred_presence_prob"] * test_preds["pred_cover_if_present"]
)
test_preds["pred_wh_area_m2"] = test_preds["pred_expected_cover"] * test_preds["valid_area_m2"]
test_preds["pred_wh_area_ha"] = test_preds["pred_wh_area_m2"] / 10_000

display(test_preds.head())

monthly_pred_summary = (
    test_preds.groupby("month", as_index=False)
    .agg(
        observed_wh_area_ha=("wh_area_ha", "sum"),
        predicted_wh_area_ha=("pred_wh_area_ha", "sum"),
        observed_mean_cover=("wh_cover", "mean"),
        predicted_mean_cover=("pred_expected_cover", "mean"),
    )
)

display(monthly_pred_summary)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly_pred_summary["month"], monthly_pred_summary["observed_wh_area_ha"], marker="o", label="Observed")
ax.plot(monthly_pred_summary["month"], monthly_pred_summary["predicted_wh_area_ha"], marker="o", label="Predicted")
ax.set_title("Observed vs predicted WH area in test months")
ax.set_xlabel("Month")
ax.set_ylabel("WH area (ha)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(test_preds["wh_cover"], test_preds["pred_expected_cover"], s=8, alpha=0.4)
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_title("Observed vs predicted expected WH cover")
ax.set_xlabel("Observed WH cover")
ax.set_ylabel("Predicted expected WH cover")
ax.grid(True, alpha=0.3)
plt.show()

## 15. Map mean observed WH cover by grid cell

This checks whether the spatial pattern looks ecologically sensible.

Cells with consistently high cover should generally correspond to sheltered littoral/embayment zones if the classification and grid design are working.

In [ ]:
cell_summary = (
    panel.groupby("grid_id", as_index=False)
    .agg(
        mean_wh_cover=("wh_cover", "mean"),
        max_wh_cover=("wh_cover", "max"),
        occurrence_rate=("wh_present", "mean"),
        mean_wh_area_ha=("wh_area_ha", "mean"),
        n_months=("month", "nunique"),
    )
)

grid_summary = grid.merge(cell_summary, on="grid_id", how="left")

ax = grid_summary.to_crs("EPSG:4326").plot(
    column="mean_wh_cover",
    figsize=(9, 7),
    legend=True,
    missing_kwds={"color": "lightgrey", "label": "No valid observations"},
)
ax.set_title("Mean observed WH cover by grid cell")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

ax = grid_summary.to_crs("EPSG:4326").plot(
    column="occurrence_rate",
    figsize=(9, 7),
    legend=True,
    missing_kwds={"color": "lightgrey", "label": "No valid observations"},
)
ax.set_title("WH occurrence rate by grid cell")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()

## 16. Export outputs

In [ ]:
panel_csv = OUTPUT_DIR / f"wh_spatial_panel_{CELL_SIZE_M}m_{TEST_START}_to_{TEST_END}.csv"
test_preds_csv = OUTPUT_DIR / f"wh_spatial_panel_predictions_{CELL_SIZE_M}m_{TEST_START}_to_{TEST_END}.csv"
monthly_summary_csv = OUTPUT_DIR / f"wh_monthly_summary_{CELL_SIZE_M}m_{TEST_START}_to_{TEST_END}.csv"
coef_csv = OUTPUT_DIR / f"wh_model_coefficients_{CELL_SIZE_M}m_{TEST_START}_to_{TEST_END}.csv"
grid_summary_gpkg = OUTPUT_DIR / f"wh_grid_summary_{CELL_SIZE_M}m_{TEST_START}_to_{TEST_END}.gpkg"

panel.to_csv(panel_csv, index=False)
test_preds.to_csv(test_preds_csv, index=False)
monthly_summary.to_csv(monthly_summary_csv, index=False)

coef_out = presence_coef.merge(cover_coef, on="feature", how="outer")
coef_out.to_csv(coef_csv, index=False)

grid_summary.to_file(grid_summary_gpkg, layer="grid_summary", driver="GPKG")

print("Saved:")
print(panel_csv)
print(test_preds_csv)
print(monthly_summary_csv)
print(coef_csv)
print(grid_summary_gpkg)

## 17. Interpretation checklist

Use this checklist before treating the results as environmental inference.

### Raster/classification checks

- Confirm that `WH_CLASS_VALUES` correctly identifies water hyacinth.
- Inspect several classified rasters visually.
- Check that missing/cloudy months are not being interpreted as true WH absence.
- Compare S1-derived and S2-derived classifications separately if both are present.

### Panel-design checks

- Test 500 m vs 1 km cells.
- Consider restricting the grid to a littoral/water mask rather than the full bounding box.
- Avoid pixel-level modelling because this creates extreme pseudo-replication.
- Check whether high-cover cells correspond to plausible embayments, sheltered shores or river-mouth zones.

### Modelling checks

- Use temporally blocked validation.
- Consider spatially blocked validation for final results.
- Check autocorrelation in residuals.
- Check collinearity among rainfall, turbidity, chlorophyll-a and lake level.
- Treat coefficients as associations, not proof of causation.
- For the final dissertation, consider a GAM, GLMM, hurdle model or Bayesian spatio-temporal model if time allows.

### Recommended next upgrade

The most useful next addition is a proper spatial-covariate table for each grid cell:

- distance to nearest river mouth;
- distance to shore;
- mean depth;
- fetch/shelter index;
- bay/embayment identity;
- wetland adjacency.

That would turn this from a mostly technical spatial-panel prototype into a genuinely ecological spatio-temporal model.